## Introduction 
In this notebook I will be developing an ANN (Artificial Neural Network) model which will classify book reviews according to sentiment.
### Dataset
The data i will be using is from kaggle, it is Goodreads book reviews. It contains user reviews with star ratings (1-5).

### Tools & Libraries
I will be using:
| Library | Purpose |
|---|---|
| **pandas** | Loading and cleaning the dataset |
| **re** | Text preprocessing (removing punctuation) |
| **collections.Counter** | Counting word frequencies to build vocabulary |
| **scikit-learn** | Train/test split |
| **torch** | Building and training the neural network |
| **torch.nn** | Model architecture (Linear layers, ReLU, Embedding) |
| **torch.utils.data** | DataLoader and TensorDataset for batching |

## Installing the data set that I'll use to train the model.
For this I am using Kaggle which I need to install on my PC and then log in.
After that I explore the data.

In [1]:
pip install kaggle

Note: you may need to restart the kernel to use updated packages.


In [2]:
!kaggle auth login

You are already logged-in to Kaggle as [dimidata].
Please use the --force flag to override.


In [3]:
import kaggle
kaggle.api.dataset_download_files(
    'pypiahmad/goodreads-book-reviews',
    path='./data',
    unzip=True)

Dataset URL: https://www.kaggle.com/datasets/pypiahmad/goodreads-book-reviews


KeyboardInterrupt: 

In [ ]:
import os

size_gb = os.path.getsize('./data/goodreads_reviews_dedup.json') / (1024**3)
print(f"{size_gb:.2f} GB")

In [27]:
import pandas as pd
df = pd.read_json(
    './data/goodreads_reviews_dedup.json',
    lines=True,
    nrows=50000)
df.head()

,user_id,book_id,review_id,rating,review_text,date_added,date_updated,read_at,started_at,n_votes,n_comments
0,8842281e1d1347389f2ab93d60773d4d,24375664,5cd416f3efc3f944fce4ce2db2290d5e,5,Mind blowingly cool. Best science fiction I've...,Fri Aug 25 13:55:02 -0700 2017,Mon Oct 09 08:55:59 -0700 2017,Sat Oct 07 00:00:00 -0700 2017,Sat Aug 26 00:00:00 -0700 2017,16,0
1,8842281e1d1347389f2ab93d60773d4d,18245960,dfdbb7b0eb5a7e4c26d59a937e2e5feb,5,This is a special book. It started slow for ab...,Sun Jul 30 07:44:10 -0700 2017,Wed Aug 30 00:00:26 -0700 2017,Sat Aug 26 12:05:52 -0700 2017,Tue Aug 15 13:23:18 -0700 2017,28,1
2,8842281e1d1347389f2ab93d60773d4d,6392944,5e212a62bced17b4dbe41150e5bb9037,3,I haven't read a fun mystery book in a while a...,Mon Jul 24 02:48:17 -0700 2017,Sun Jul 30 09:28:03 -0700 2017,Tue Jul 25 00:00:00 -0700 2017,Mon Jul 24 00:00:00 -0700 2017,6,0
3,8842281e1d1347389f2ab93d60773d4d,22078596,fdd13cad0695656be99828cd75d6eb73,4,"Fun, fast paced, and disturbing tale of murder...",Mon Jul 24 02:33:09 -0700 2017,Sun Jul 30 10:23:54 -0700 2017,Sun Jul 30 15:42:05 -0700 2017,Tue Jul 25 00:00:00 -0700 2017,22,4
4,8842281e1d1347389f2ab93d60773d4d,6644782,bd0df91c9d918c0e433b9ab3a9a5c451,4,A fun book that gives you a sense of living in...,Mon Jul 24 02:28:14 -0700 2017,Thu Aug 24 00:07:20 -0700 2017,Sat Aug 05 00:00:00 -0700 2017,Sun Jul 30 00:00:00 -0700 2017,8,0


After exploring the data I have decided to train the model to predict how many stars will a reviewer give depending on the text they have written.

In [28]:
print(df.shape)
print(df["rating"].value_counts())
print(df[['review_text', 'rating']].isnull().sum())

(50000, 11)
rating
4    17636
5    13972
3    11052
2     3941
0     1966
1     1433
Name: count, dtype: int64
review_text    0
rating         0
dtype: int64


The data is very complete and there isn't any missing information. But the data is not evenly distributed. There are a lot of 4 star reviews and very few 1 star. Also, the 0 star does not reflect a rating; in good reads, 0 stars = no rating. I will delete the rows that don't have a star rating.

In [29]:
df = df[df['rating'] != 0]
df = df[['review_text', 'rating']].reset_index(drop=True)

print(df.shape)
print(df['rating'].value_counts())

(48034, 2)
rating
4    17636
5    13972
3    11052
2     3941
1     1433
Name: count, dtype: int64


There are a lot more positive reviews than negatives; after some research, I have decided to create 3 different categories: "negative", "neutral", and "positive".


In [30]:
def to_label(r):
    if r >= 4:   return 2   # positive
    elif r == 3: return 1   # neutral
    else:        return 0   # negative

df['label'] = df['rating'].apply(to_label)

print(df['label'].value_counts())

label
2    31608
1    11052
0     5374
Name: count, dtype: int64


Now we have the reves stored as labels but after that we need to tokenize the text so the model can actually use it for training. Seeing the project does not expect a deep semantic understanding, I'll try to use pythons build in library re. (Regular expression operations) to tokenize the text. 
First we will standerdise the text, replacing everything that is not a letter or a space by an empty string "", then will will make the whole text lower case. 
Then we count all the words in the texts, we build a dictionnary, then we give every word a nuber.
For the training well need to have revews with the same word leght so if the revew is shorter we just put 0s utill we have the same lenght, if a word is not in the 10 000 more used we label it as unknown.

In [31]:
from collections import Counter
import re

def tokenize(text):
    # lowercase and split on non-letters
    return re.sub(r'[^a-z\s]', '', text.lower()).split()
    # Count every word across all reviews
word_counts = Counter()
for text in df['review_text']:
    word_counts.update(tokenize(text))

# Keep only the 10,000 most common words
VOCAB_SIZE = 10_000
vocab = {word: idx+2 for idx, (word, _) in enumerate(word_counts.most_common(VOCAB_SIZE))}
vocab['<PAD>'] = 0   #  padding token
vocab['<UNK>'] = 1   # unknown words not in vocabulary

print(f"Vocabulary size: {len(vocab)}")
print(f"Sample words: {list(vocab.items())[:10]}")

Vocabulary size: 10002
Sample words: [('the', 2), ('and', 3), ('a', 4), ('to', 5), ('i', 6), ('of', 7), ('is', 8), ('in', 9), ('it', 10), ('this', 11)]


 We have a dictionnary but the next step uses the vocabulary to convert the strings in the revews into lists of numbers that PyTorch can actually work with. Now every revew is encoded according to the dictionnary. 
We define X as the input, the text revew and y as the output, the label.


In [36]:
MAX_LEN = 200  # max words per review

def encode(text):
    tokens = tokenize(text)[:MAX_LEN]        # truncate long reviews
    ids = [vocab.get(t, 1) for t in tokens]  # 1 = <UNK> for unknown words
    ids += [0] * (MAX_LEN - len(ids))        # pad shorter reviews
    return ids

# Encode all reviews
X = [encode(text) for text in df['review_text']]
y = df['label'].tolist()

print(f"First encoded review (first 20 tokens): {X[0][:20]}")
print(f"First label: {y[0]}")

First encoded review (first 20 tokens): [314, 1, 957, 166, 965, 419, 197, 26, 9, 62, 65, 6, 45, 90, 35, 2, 1023, 7, 2, 665]
First label: 2


Now we can finly move on to training the model. We now have transform the python list in PyTorch tensor wich correspond to x and y.

In [37]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Convert to tensors
X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.long)

print(f"X shape: {X_tensor.shape}")
print(f"y shape: {y_tensor.shape}")

X shape: torch.Size([48034, 200])
y shape: torch.Size([48034])


When we train a model we need to separate tha data into training and testing data, for that we use sklearn.

In [38]:
from sklearn.model_selection import train_test_split

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor,
    test_size=0.2,
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")

Training samples: 38427
Test samples:     9607


Now we need to wrap these into DataLoaders wich is PyTorch's way of feeding data to the model in small batches during training.

In [39]:
# Wrap in TensorDataset
train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test, y_test)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches:     {len(test_loader)}")

Training batches: 1201
Test batches:     301


In [41]:
import torch.nn as nn

class ReviewClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE + 2, 64)  # word to dense vector
        self.fc1 = nn.Linear(64 * MAX_LEN, 128)            # first linear layer
        self.relu = nn.ReLU()                               # non-linear activation
        self.fc2 = nn.Linear(128, 3)                       # output layer (3 classes)

    def forward(self, x):
        x = self.embedding(x)          # [batch, 200] → [batch, 200, 64]
        x = x.view(x.size(0), -1)      # flatten → [batch, 12800]
        x = self.fc1(x)                # [batch, 12800] → [batch, 128]
        x = self.relu(x)               # add non-linearity
        x = self.fc2(x)                # [batch, 128] → [batch, 3]
        return x

model = ReviewClassifier()
print(model)

ReviewClassifier(
  (embedding): Embedding(10002, 64)
  (fc1): Linear(in_features=12800, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)


For the model to recognise words we need to embedd them, so they can becom vectors (random) that the model can reconise. As the model learns words with similar meaning will have simicalr vectors. here each word index is replaced tith 64 numbers. 

In [14]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss() #appropriate for classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()           # reset gradients
        predictions = model(X_batch)    # forward pass
        loss = criterion(predictions, y_batch)  # calculate loss
        loss.backward()                 # backward pass
        optimizer.step()                # update weights
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS}  loss: {avg_loss:.4f}")


Epoch 1/5  loss: 0.8838
Epoch 2/5  loss: 0.6409
Epoch 3/5  loss: 0.5678
Epoch 4/5  loss: 0.4683
Epoch 5/5  loss: 0.3941


In [ ]:
we use othe optimizer to update the weights (adam wich was reccomended) we do 5 epochs so that the model dost't do overfitting. After the test i may add epchs.

In [15]:
model.eval()  # switch to evaluation mode
correct = 0
total = 0

with torch.no_grad():  # don't compute gradients during evaluation
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        predicted_labels = predictions.argmax(dim=1)  # pick highest scoring class
        correct += (predicted_labels == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct / total
print(f"Test accuracy: {accuracy:.2%}")

Test accuracy: 68.09%


In [ ]:
the model does't have the best accuracy, we should ivastigate why.

In [16]:
# Check predictions per class
from collections import Counter

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        predicted_labels = predictions.argmax(dim=1)
        all_preds.extend(predicted_labels.tolist())
        all_labels.extend(y_batch.tolist())

print("Predicted distribution:", Counter(all_preds))
print("Actual distribution:   ", Counter(all_labels))

Predicted distribution: Counter({2: 1644, 1: 235, 0: 45})
Actual distribution:    Counter({2: 1339, 1: 417, 0: 168})


The model is predicting mainly positive revews becouse the date hase way more positive then negative revews. To fix this problem we can tell the loss function to penalize mistakes on rare classes more heavily

In [17]:
# Calculate class weights (inverse of frequency)
class_counts = torch.tensor([929, 2033, 6655], dtype=torch.float)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()  # normalize

print("Class weights:", class_weights)

# Recreate loss function with weights
criterion = nn.CrossEntropyLoss(weight=class_weights)

Class weights: tensor([0.6263, 0.2862, 0.0874])


Here we have the highest penalty for predicting badly the negative revews (0.6263) and the lowest for the positive ones (0.0874) 

In [19]:
model = ReviewClassifier()  # reset the model
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/10  loss: {total_loss/len(train_loader):.4f}")

Epoch 1/10  loss: 1.2243
Epoch 2/10  loss: 0.9312
Epoch 3/10  loss: 0.8411
Epoch 4/10  loss: 0.7218
Epoch 5/10  loss: 0.5933
Epoch 6/10  loss: 0.4985
Epoch 7/10  loss: 0.3764
Epoch 8/10  loss: 0.3055
Epoch 9/10  loss: 0.2606
Epoch 10/10  loss: 0.1859


I have alsoo added 5 more epochs becouse the loss is dropping steadily so more epochs will help.

In [21]:
# Check accuracy
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        predicted_labels = predictions.argmax(dim=1)
        correct += (predicted_labels == y_batch).sum().item()
        total += y_batch.size(0)
        all_preds.extend(predicted_labels.tolist())
        all_labels.extend(y_batch.tolist())

print(f"Test accuracy: {correct/total:.2%}")
print("Predicted distribution:", Counter(all_preds))
print("Actual distribution:   ", Counter(all_labels))

Test accuracy: 54.83%
Predicted distribution: Counter({2: 971, 1: 850, 0: 103})
Actual distribution:    Counter({2: 1339, 1: 417, 0: 168})


In [ ]:
We can see that the accuracy droped, we overdid the correction, wich meant the penalty is too big. We can again ajust the weight but with less.

In [22]:
# Softer weights, less aggressive
class_weights = torch.tensor([2.0, 1.0, 0.5], dtype=torch.float)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Retrain
model = ReviewClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/10  loss: {total_loss/len(train_loader):.4f}")

Epoch 1/10  loss: 1.1148
Epoch 2/10  loss: 0.8942
Epoch 3/10  loss: 0.8105
Epoch 4/10  loss: 0.7356
Epoch 5/10  loss: 0.6438
Epoch 6/10  loss: 0.5599
Epoch 7/10  loss: 0.4733
Epoch 8/10  loss: 0.4018
Epoch 9/10  loss: 0.3637
Epoch 10/10  loss: 0.3066


In [23]:
# Check accuracy
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        predicted_labels = predictions.argmax(dim=1)
        correct += (predicted_labels == y_batch).sum().item()
        total += y_batch.size(0)
        all_preds.extend(predicted_labels.tolist())
        all_labels.extend(y_batch.tolist())

print(f"Test accuracy: {correct/total:.2%}")
print("Predicted distribution:", Counter(all_preds))
print("Actual distribution:   ", Counter(all_labels))

Test accuracy: 61.54%
Predicted distribution: Counter({2: 1198, 1: 677, 0: 49})
Actual distribution:    Counter({2: 1339, 1: 417, 0: 168})


In [ ]:
Now the model is overpredictiong nutual revews and ignoring negative ones. we can again adjust the weight (i am using the same code to check accuracy)  

In [24]:
class_weights = torch.tensor([3.0, 0.7, 0.5], dtype=torch.float)
criterion = nn.CrossEntropyLoss(weight=class_weights)

model = ReviewClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/10  loss: {total_loss/len(train_loader):.4f}")

Epoch 1/10  loss: 1.2515
Epoch 2/10  loss: 0.8202
Epoch 3/10  loss: 0.7067
Epoch 4/10  loss: 0.5874
Epoch 5/10  loss: 0.4882
Epoch 6/10  loss: 0.4152
Epoch 7/10  loss: 0.3317
Epoch 8/10  loss: 0.2746
Epoch 9/10  loss: 0.2169
Epoch 10/10  loss: 0.1924


In [26]:
# Check accuracy
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        predicted_labels = predictions.argmax(dim=1)
        correct += (predicted_labels == y_batch).sum().item()
        total += y_batch.size(0)
        all_preds.extend(predicted_labels.tolist())
        all_labels.extend(y_batch.tolist())

print(f"Test accuracy: {correct/total:.2%}")
print("Predicted distribution:", Counter(all_preds))
print("Actual distribution:   ", Counter(all_labels))

Test accuracy: 60.55%
Predicted distribution: Counter({2: 1283, 1: 347, 0: 294})
Actual distribution:    Counter({2: 1339, 1: 417, 0: 168})


Seeing that chainging the weight hasn't worked i will reuse the code to but add more data. 

Atre reloding the data here well train the model with 50 000 examples using the same code 

In [42]:
criterion = nn.CrossEntropyLoss()  # no weights this time
model = ReviewClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/10  loss: {total_loss/len(train_loader):.4f}")

Epoch 1/10  loss: 0.8687
Epoch 2/10  loss: 0.7051
Epoch 3/10  loss: 0.5587
Epoch 4/10  loss: 0.4794
Epoch 5/10  loss: 0.4090
Epoch 6/10  loss: 0.3508
Epoch 7/10  loss: 0.3195
Epoch 8/10  loss: 0.2719
Epoch 9/10  loss: 0.2403
Epoch 10/10  loss: 0.2086


In [43]:
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        predicted_labels = predictions.argmax(dim=1)
        correct += (predicted_labels == y_batch).sum().item()
        total += y_batch.size(0)
        all_preds.extend(predicted_labels.tolist())
        all_labels.extend(y_batch.tolist())

print(f"Test accuracy: {correct/total:.2%}")
print("Predicted distribution:", Counter(all_preds))
print("Actual distribution:   ", Counter(all_labels))

Test accuracy: 65.36%
Predicted distribution: Counter({2: 6544, 1: 2135, 0: 928})
Actual distribution:    Counter({2: 6376, 1: 2200, 0: 1031})


The data is better distributed with more examples but it doesn't go over 70% but we'll continue and try to test the algorithm with some sentences. 

In [44]:
def predict(review_text):
    encoded = encode(review_text)
    tensor = torch.tensor([encoded], dtype=torch.long)
    model.eval()
    with torch.no_grad():
        output = model(tensor)
        label = output.argmax(dim=1).item()
    labels = {0: "negative", 1: "neutral", 2: "positive"}
    return labels[label]


print(predict("This book was an absolute masterpiece, deeply moving and beautifully written"))
print(predict("Boring and tedious, I couldn't finish it"))
print(predict("It was okay, nothing special but not terrible either"))

positive
negative
negative


In [45]:
# More obviously neutral
print(predict("An average book, some parts were good some were bad"))

# Strong positive
print(predict("One of the greatest novels I have ever read, a true classic"))

# Strong negative  
print(predict("Terrible writing, shallow characters, complete waste of time"))

# Tricky — positive words but negative meaning
print(predict("I wanted to love this book but it was a disappointment"))

negative
positive
negative
negative


the model seems good at predicting the two extrams but not the nutual revews. 
We'll try it with real revews:

In [50]:
# 3 star
print(predict("This novel has a fascinating premise. The author is a poet, and there are many delightfully beautiful turns of phrase. Overall, this was slow. The characters are one dimensional and feel in most cases like literal stereotypes. I think if this was written in the 1920s, it would be fabulous, but as written in the 2020s and set in modern times, it falls flat. Honestly, this feels like an idea for a story that didn't get seasoned well or cooked long enough"))

neutral


In [51]:
# 1 star 
print(predict("In my nonfiction phase during the year, I grabbed this one and after finishing it, regretted its purchase. The book is about medical use of corpses and the human body, present-day and in the past. The subject matter is extremely interesting, and some of the methods, tests, and history behind human body experiments is worth the read. The book makes you want to be an organ donor, or want to donate your body to medical science. The problem is that the author is one of the WORST writers I have ever read to the extent that every time I picked up the book I got angry. I only finished the book because my OCD made me finish it because I’d already started it. The two irritating aspects of the book are: 1) Roach would spend a few pages describing something fascinating and then ruin it all by throwing in the snarkiest comment imaginable. For example, she’d discuss how feet are used by scientists, and then throw in a comment about her stinky socks. 2) A few years ago, a friend saw a movie about the roads to concentration camps at the Tribeca Film Festival that was atrocious because the director stuck himself into the film and made himself part of the story. That’s what this author does for the whole friggin’ book. Just awful."))

positive


In [54]:
#5 star
print(predict("""The best book about shit you will ever read. Genuinely, this is very well-written and crafted with intention to cover pretty much every common (and uncommon) question you might have about your gut health. I picked this up for two reasons. First, I have become extremely paranoid about colon cancer; it seems like it is everywhere with young, big name celebs passing from it. With more and more news stories about its increasing diagnoses among young people, people in my weightlifting class are swapping fiber tips... The colon is the thing right now. So get educated! Second, John Green recommended it, and, as you can maybe tell from the reviews, Nerdfighters really will take any book rec.

I highly recommend this as a straightforward, dense but approachable guide to keeping your body healthy. She covers topics from color, formation, fiber intake, preventing the runs during your runs (lol), the brain-gut connection, and more. I do wish she had covered a bit more around cancer screenings (like, many practitioners skepticism around send-away tests, etc.) but I also have a specific paranoia around this. Either way, this is where to start to make sure your insides are running properly, highly recommend on audio."""))

positive


## Conclusion

The ANN model allows us to generate predictions by learning from labeled examples, 
a form of supervised learning. With enough data and training, this type of model can 
become better and more useful. It also can be used in many different domains. Here, 
the project does sentiment analysis which is useful in many fields.

The project has allowed me to practically implement the notions I have gained during 
the course, but also combine them with what I have learned in natural language processing.

Dealing with the problems that we saw during the project allowed me to explore and gain 
more insight into how ANN models work, which allowed me to also see the limitations of 
the methods I have used. For future work, it will be useful to use transformers which 
are much more powerful when it comes to treating textual data.

### Limitations

| Limitation | Impact |
|---|---|
| Bag of words cannot capture context | Cannot handle irony or sarcasm |
| Class imbalance in the data | Solutions exist but are not 100% reliable |
| Word order not preserved | Mixed sentiment reviews misclassified |

### Results

> **Final model accuracy: ~65%** on a 3-class sentiment prediction task (positive, neutral, negative)

### Future Work
Using **transformers** would address most of these limitations as they are much more 
powerful when it comes to treating textual data.
